# Практическая работа № 7. Случайный лес
**Дисциплина:** «Машинное обучение». **Уровень B.**

**Задача.** Модель оттока абонентов мобильного оператора на основе случайного леса и выяснение, какие признаки действительно влияют на уход.

> Датасет `churn_synthetic.csv` синтетический: 3000 абонентов, сгенерированы программой. Это не реальные данные О! или другого оператора, названия компаний — только контекст задачи.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, f1_score

SEED = 42
CSV = "churn_synthetic.csv"

def make_standin(n=3000, seed=SEED):
    """Запасной генератор — ТОЛЬКО чтобы ноутбук запускался без файла.
    Для сдачи используйте настоящий churn_synthetic.csv."""
    r = np.random.default_rng(seed)
    d = pd.DataFrame({
        "subscriber_code": [f"S{i:05d}" for i in range(n)],
        "region": r.choice(["Бишкек", "Ош", "Чуй", "Иссык-Куль", "Джалал-Абад", "Нарын"], n),
        "tariff": r.choice(["Базовый", "Смарт", "Безлимит", "Семейный"], n, p=[.3, .3, .25, .15]),
        "tenure_months": r.integers(1, 73, n),
        "internet_gb": r.gamma(3, 4, n).round(1),
        "support_calls_3m": r.poisson(1.2, n),
        "autopay": r.integers(0, 2, n),
        "days_since_topup": r.integers(0, 60, n),
        "topups_3m": r.poisson(3, n),
        "dropped_calls_pct": r.gamma(2, 1.2, n).round(2),
        "balance_som": r.normal(150, 120, n).round(1),
    })
    base = d.tariff.map({"Базовый": 250, "Смарт": 450, "Безлимит": 700, "Семейный": 900})
    d["monthly_charge_som"] = (base + r.normal(0, 40, n)).round(0)
    z = (-1.6 + 0.35 * d.support_calls_3m + 0.05 * d.days_since_topup - 0.03 * d.tenure_months
         - 0.8 * d.autopay + 0.25 * d.dropped_calls_pct - 0.2 * d.topups_3m
         + 0.001 * (d.monthly_charge_som - 500) + r.normal(0, .8, n))
    d["churn"] = (r.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    return d

STANDIN = not os.path.exists(CSV)
df = make_standin() if STANDIN else pd.read_csv(CSV)
print("ЗАПАСНЫЕ ДАННЫЕ (файл не найден)!" if STANDIN else "Загружен", CSV, df.shape)
print("Доля оттока:", df.churn.mean().round(3))
df.head()

## Подготовка
`subscriber_code` — идентификатор, а не признак: его исключаем из `X`, иначе модель может «запомнить» абонентов. Разбиение и кодирование — как в условии.

In [ ]:
CAT = ["region", "tariff"]
X = df.drop(columns=["churn", "subscriber_code"])
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

def make_pipe(features=None, **rf_params):
    feats = list(features) if features is not None else list(X.columns)
    cat = [c for c in feats if c in CAT]
    num = [c for c in feats if c not in CAT]
    pre = ColumnTransformer([("ohe", OneHotEncoder(handle_unknown="ignore"), cat)] if cat else [],
                            remainder="passthrough")
    # ColumnTransformer без transformers не работает, поэтому при cat=[] даём passthrough
    if not cat:
        pre = "passthrough"
    rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, **rf_params)
    return Pipeline([("pre", pre), ("rf", rf)])

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
print(X_train.shape, X_test.shape)

## 1. Подбор гиперпараметров (GridSearchCV)

In [ ]:
param_grid = {
    "rf__n_estimators": [100, 300],
    "rf__max_depth": [None, 6, 10],
    "rf__min_samples_leaf": [1, 5, 20],
    "rf__max_features": ["sqrt", 0.5],
}
gs = GridSearchCV(make_pipe(), param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
gs.fit(X_train, y_train)
print("Лучшие параметры:", gs.best_params_)
print("Лучший CV ROC-AUC: %.4f" % gs.best_score_)
res = pd.DataFrame(gs.cv_results_).sort_values("rank_test_score")
res[["params", "mean_test_score", "std_test_score"]].head(8)

## 2. Настроенная модель против модели по умолчанию (test)

In [ ]:
default = make_pipe().fit(X_train, y_train)
tuned = gs.best_estimator_

rows = []
for name, m in [("По умолчанию", default), ("Настроенная", tuned)]:
    p = m.predict_proba(X_test)[:, 1]
    rows.append({"Модель": name,
                 "ROC-AUC": roc_auc_score(y_test, p),
                 "F1 (порог 0.5)": f1_score(y_test, (p >= 0.5).astype(int))})
cmp_ = pd.DataFrame(rows).set_index("Модель").round(4)
cmp_

In [ ]:
d_auc = cmp_.loc["Настроенная", "ROC-AUC"] - cmp_.loc["По умолчанию", "ROC-AUC"]
d_f1 = cmp_.loc["Настроенная", "F1 (порог 0.5)"] - cmp_.loc["По умолчанию", "F1 (порог 0.5)"]
print(f"ΔROC-AUC = {d_auc:+.4f},  ΔF1 = {d_f1:+.4f}")
if d_auc * d_f1 < 0:
    print("Метрики изменились в РАЗНЫЕ стороны — см. объяснение ниже.")

**Почему метрики могут разойтись.** ROC-AUC оценивает *ранжирование* по вероятностям и не зависит от порога, а GridSearchCV оптимизировал именно его. F1 считается при фиксированном пороге 0.5 по классам. Ограничение `max_depth` и рост `min_samples_leaf` «сглаживают» вероятности: лес реже выдаёт значения выше 0.5 для редкого класса (отток), поэтому полнота падает, и F1 при пороге 0.5 может ухудшиться, даже если порядок абонентов по риску стал лучше (AUC вырос). Для F1 надо отдельно подбирать порог или оптимизировать F1 в `scoring`.

*(Ниже в выводе укажите, что получилось у вас: если метрики изменились в одну сторону — это объяснение не нужно, достаточно написать, что настройка помогла/не помогла.)*

## 3. OOB-оценка по числу деревьев

In [ ]:
best_rf = {k.replace("rf__", ""): v for k, v in gs.best_params_.items() if k != "rf__n_estimators"}
ns = [10, 25, 50, 100, 200, 400]
oob = []
for n in ns:
    p = make_pipe(n_estimators=n, oob_score=True, **best_rf).fit(X_train, y_train)
    oob.append(p.named_steps["rf"].oob_score_)   # OOB accuracy
plt.figure(figsize=(6, 3.8))
plt.plot(ns, oob, "o-")
plt.xscale("log"); plt.xticks(ns, ns)
plt.xlabel("n_estimators"); plt.ylabel("OOB accuracy"); plt.grid(alpha=.3)
plt.title("OOB-оценка (accuracy) в зависимости от числа деревьев")
plt.show()
pd.Series(oob, index=ns, name="OOB accuracy").round(4)

**Выбор числа деревьев.** По графику оценка быстро растёт при малом числе деревьев и затем выходит на плато. Выбираем наименьшее `n_estimators`, после которого прирост OOB становится меньше шума (порядка нескольких тысячных): дальнейшее увеличение только удлиняет обучение и предсказание. *(Назовите конкретное число по вашему графику.)*

Примечание: `oob_score_` в sklearn — это accuracy. При дисбалансе классов она менее информативна, чем ROC-AUC, поэтому выбор числа деревьев подтверждаем кросс-валидационным AUC.

## 4. Эксперимент с удалением признаков

In [ ]:
# Важность — по настроенной модели на train, суммируем по исходным признакам (OHE-колонки объединяем)
rf = tuned.named_steps["rf"]
names = tuned.named_steps["pre"].get_feature_names_out()
imp = pd.Series(rf.feature_importances_, index=names)

def orig(n):
    n = n.split("__", 1)[1]
    for c in CAT:
        if n.startswith(c + "_"):
            return c
    return n

importance = imp.groupby(orig).sum().sort_values(ascending=False)
print(importance.round(4))
order = list(importance.index[::-1])   # от наименее важного к наиболее важному

In [ ]:
params = {k.replace("rf__", ""): v for k, v in gs.best_params_.items()}
feats = list(X.columns)
log = []
s = cross_val_score(make_pipe(feats, **params), X_train[feats], y_train, scoring="roc_auc", cv=cv, n_jobs=-1)
log.append({"Удалено": "— (все признаки)", "Осталось": len(feats), "mean": s.mean(), "std": s.std()})
for f in order[:-1]:                       # последний признак не удаляем
    feats.remove(f)
    s = cross_val_score(make_pipe(feats, **params), X_train[feats], y_train, scoring="roc_auc", cv=cv, n_jobs=-1)
    log.append({"Удалено": f, "Осталось": len(feats), "mean": s.mean(), "std": s.std()})
abl = pd.DataFrame(log)
abl["ROC-AUC (cv)"] = abl["mean"].round(4).astype(str) + " ± " + abl["std"].round(4).astype(str)
abl[["Удалено", "Осталось", "ROC-AUC (cv)"]]

In [ ]:
plt.figure(figsize=(8, 4))
plt.errorbar(range(len(abl)), abl["mean"], yerr=abl["std"], fmt="o-", capsize=3)
plt.xticks(range(len(abl)), abl["Удалено"], rotation=60, ha="right")
base_m, base_s = abl.loc[0, "mean"], abl.loc[0, "std"]
plt.axhspan(base_m - base_s, base_m + base_s, alpha=.15, label="базовый ± std")
plt.ylabel("ROC-AUC (5-fold, mean ± std)"); plt.legend(); plt.grid(alpha=.3)
plt.title("Последовательное удаление наименее важных признаков")
plt.tight_layout(); plt.show()

## 5. Вывод: что можно исключить

In [ ]:
# Различие считаем реальным, только если оно больше разброса между фолдами:
# падение mean относительно базового > std базового.
drop_ok = []
for i in range(1, len(abl)):
    if base_m - abl.loc[i, "mean"] <= base_s:
        drop_ok.append(abl.loc[i, "Удалено"])
    else:
        break   # дальше качество уже реально падает
print("Можно исключить без реальной потери качества:", drop_ok)
print("Оставить:", [f for f in importance.index if f not in drop_ok])
print(f"Базовый AUC {base_m:.4f} ± {base_s:.4f}")

final = make_pipe([f for f in X.columns if f not in drop_ok], **params).fit(X_train[[f for f in X.columns if f not in drop_ok]], y_train)
pf = final.predict_proba(X_test[[f for f in X.columns if f not in drop_ok]])[:, 1]
print("Test ROC-AUC без этих признаков: %.4f (с ними: %.4f)" % (roc_auc_score(y_test, pf), cmp_.loc["Настроенная", "ROC-AUC"]))

### Вывод на полстраницы (заполните по своим результатам)
1. **Данные.** Датасет синтетический (3000 абонентов), не реальные данные О!. `subscriber_code` исключён из признаков как идентификатор.
2. **Настройка.** Лучшие параметры: … ; CV ROC-AUC = … Сравнение на test: настроенная модель … (ROC-AUC …, F1 …). Если метрики разошлись — объяснение выше (AUC не зависит от порога, F1 при 0.5 чувствителен к «сглаживанию» вероятностей).
3. **Число деревьев.** По OOB-кривой выбрано … деревьев: дальше прирост меньше шума.
4. **Признаки.** Наиболее важные: …  Можно не собирать/исключить: … (падение ROC-AUC в пределах std между фолдами, т.е. различие не реально).
5. **Рекомендация маркетингу.** Продолжать собирать …; сбор … можно прекратить. Оговорка: важность по impurity смещена в пользу непрерывных признаков и признаков с большим числом значений, а зависимости корреляционные — причинность не доказана.